In [4]:
import os, random, math
from PIL import Image, ImageDraw, ImageFilter

SIZE = 128
N_PER_CLASS = 500
classes = ["Circle", "Square", "Triangle"]

def rand_color():
    return tuple(random.randint(0, 255) for _ in range(3))

def make_image(shape):
    bg = rand_color()
    fg = rand_color()
    while sum(abs(a - b) for a, b in zip(fg, bg)) < 150:
        fg = rand_color()
    img = Image.new("RGB", (SIZE, SIZE), bg)
    d = ImageDraw.Draw(img)

    cx, cy = random.randint(50, 78), random.randint(50, 78)
    r = random.randint(25, 45)
    angle = random.uniform(0, 2 * math.pi)
    filled = random.random() < 0.7
    fill, outline = (fg, None) if filled else (None, fg)

    if shape == "Circle":
        d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=fill, outline=outline, width=4)
    else:
        n = 4 if shape == "Square" else 3
        pts = [(cx + r * math.cos(angle + 2 * math.pi * i / n),
                cy + r * math.sin(angle + 2 * math.pi * i / n)) for i in range(n)]
        d.polygon(pts, fill=fill, outline=outline, width=4)

    if random.random() < 0.5:
        img = img.filter(ImageFilter.GaussianBlur(random.uniform(0, 1.5)))
    return img

for c in classes:
    os.makedirs(f"dataset/{c}", exist_ok=True)
    for i in range(N_PER_CLASS):
        make_image(c).save(f"dataset/{c}/{c}_{i}.png")

print("Done:", {c: len(os.listdir(f"dataset/{c}")) for c in classes})

Done: {'Circle': 500, 'Square': 500, 'Triangle': 500}


In [5]:
import tensorflow as tf
from tensorflow.keras import layers

train_ds = tf.keras.utils.image_dataset_from_directory(
    "dataset", validation_split=0.2, subset="training",
    seed=42, image_size=(128, 128), batch_size=32)
test_ds = tf.keras.utils.image_dataset_from_directory(
    "dataset", validation_split=0.2, subset="validation",
    seed=42, image_size=(128, 128), batch_size=32)

print("Classes:", train_ds.class_names)

model = tf.keras.Sequential([
    layers.Input(shape=(128, 128, 3)),
    layers.Rescaling(1./255),
    layers.Conv2D(16, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(3)
])

model.compile(optimizer="adam",
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["accuracy"])

_ = model.fit(train_ds, validation_data=test_ds, epochs=25)

Found 1500 files belonging to 3 classes.
Using 1200 files for training.
Found 1500 files belonging to 3 classes.
Using 300 files for validation.
Classes: ['Circle', 'Square', 'Triangle']
Epoch 1/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 20s 484ms/step - accuracy: 0.4283 - loss: 1.0708 - val_accuracy: 0.5533 - val_loss: 0.9477
Epoch 2/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 23s 554ms/step - accuracy: 0.5567 - loss: 0.8986 - val_accuracy: 0.5600 - val_loss: 0.8845
Epoch 3/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 19s 490ms/step - accuracy: 0.5850 - loss: 0.8312 - val_accuracy: 0.5600 - val_loss: 0.8811
Epoch 4/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 21s 546ms/step - accuracy: 0.6325 - loss: 0.7670 - val_accuracy: 0.5867 - val_loss: 0.8418
Epoch 5/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 19s 487ms/step - accuracy: 0.6658 - loss: 0.7025 - val_accuracy: 0.5900 - val_loss: 0.8009
Epoch 6/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 18s 477ms/step - accuracy: 0.7008 - loss: 0.6376 - val_accuracy: 0.6367 - val_loss: 0.7454
Epoch 7/25
38/38 ━━━━━━━━━━━━━━━━━━━━ 1

In [6]:
loss, acc = model.evaluate(test_ds)
print(f"Model Accuracy: {acc*100:.1f}%")

model.save("shape_model.keras")

from google.colab import files
files.download("shape_model.keras")

10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 177ms/step - accuracy: 0.9200 - loss: 0.2602
Model Accuracy: 92.0%


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
from google.colab import files

!zip -r dataset.zip dataset
files.download("dataset.zip")

  adding: dataset/ (stored 0%)
  adding: dataset/Triangle/ (stored 0%)
  adding: dataset/Triangle/Triangle_165.png (deflated 10%)
  adding: dataset/Triangle/Triangle_198.png (deflated 13%)
  adding: dataset/Triangle/Triangle_185.png (deflated 13%)
  adding: dataset/Triangle/Triangle_378.png (deflated 11%)
  adding: dataset/Triangle/Triangle_253.png (deflated 13%)
  adding: dataset/Triangle/Triangle_368.png (deflated 8%)
  adding: dataset/Triangle/Triangle_469.png (deflated 9%)
  adding: dataset/Triangle/Triangle_22.png (deflated 9%)
  adding: dataset/Triangle/Triangle_447.png (deflated 23%)
  adding: dataset/Triangle/Triangle_121.png (deflated 16%)
  adding: dataset/Triangle/Triangle_127.png (deflated 23%)
  adding: dataset/Triangle/Triangle_400.png (deflated 20%)
  adding: dataset/Triangle/Triangle_403.png (deflated 27%)
  adding: dataset/Triangle/Triangle_51.png (deflated 11%)
  adding: dataset/Triangle/Triangle_207.png (deflated 5%)
  adding: dataset/Triangle/Triangle_460.png (defla

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
from google.colab import files
files.download("dataset.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>